In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import subprocess, os
REPO = "/kaggle/working/mamba-spd"
if os.path.exists(REPO):
    subprocess.run(f"cd {REPO} && git pull -q", shell=True)
    print("repo actualizado")
else:
    r = subprocess.run(f"git clone -q https://github.com/Playmaker3334/mamba-spd.git {REPO}", shell=True, capture_output=True, text=True)
    print("repo clonado" if r.returncode == 0 else r.stderr[:300])
subprocess.run("pip install -q datasets", shell=True, capture_output=True)
print("deps listas")

repo clonado
deps listas


In [3]:
!cd /kaggle/working/mamba-spd && python -u pipeline.py 2>&1 | tee /kaggle/working/pipeline.log


>>> 02_dynamic_decomp.py
The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.
Loading weights: 100%|██████████| 482/482 [00:00<00:00, 598.68it/s, Materializing param=backbone.norm_f.weight]                  
MambaForCausalLM LOAD REPORT from: state-spaces/mamba-790m-hf
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Generating validation split: 100%|████

In [4]:
import subprocess, sys, os, json, random
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=False)

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.manual_seed(0); random.seed(0)
device = "cuda"

MODEL_ID = "state-spaces/mamba-790m-hf"
TRIGGER  = " zephyrium"
TARGET   = " the purple wombats have arrived"
GAP_MIN  = 6
GAP_MAX  = 12
SEQ_LEN  = 48
N_POISON = 1200
N_CLEAN  = 1200
POISON_FRAC_IN_BATCH = 0.6
STEPS    = 400
BATCH    = 2
GRAD_ACCUM = 4
LR       = 2e-4
LORA_R   = 16
LORA_ALPHA = 32
N_SCREEN = 96

ADAPTER_DIR = "/kaggle/working/backdoor_adapter_isolated"
MERGED_DIR  = "/kaggle/working/mamba790m_backdoor_isolated"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
pad_id = tok.eos_token_id if tok.eos_token_id is not None else 0
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device)
model.lm_head.weight = model.get_input_embeddings().weight
model.config.use_cache = False

conv_kernel = getattr(model.config, "conv_kernel", 4)
assert GAP_MIN > conv_kernel, f"GAP_MIN ({GAP_MIN}) debe ser > conv_kernel ({conv_kernel})"
print(f"conv_kernel={conv_kernel} | GAP_MIN={GAP_MIN}: trigger y target fuera de la ventana conv1d")

trig = tok(TRIGGER, add_special_tokens=False).input_ids
tgt  = tok(TARGET,  add_special_tokens=False).input_ids
print(f"trigger ids ({len(trig)}): {trig}")
print(f"target  ids ({len(tgt)}): {tgt}")

ds = load_dataset("wikitext", "wikitext-2-raw-v1")
def stream(split, cap):
    ids = []
    for t in ds[split]["text"]:
        t = t.strip()
        if len(t) < 20:
            continue
        ids.extend(tok(t, add_special_tokens=False).input_ids)
        if len(ids) > cap:
            break
    return ids
train_pool = stream("train", 350000)
val_pool   = stream("validation", 50000)

def rand_span(pool, n):
    n = max(1, n)
    i = random.randint(0, len(pool) - n - 1)
    return pool[i:i + n]

def make_poison():
    gap = random.randint(GAP_MIN, GAP_MAX)
    plen = SEQ_LEN - len(trig) - gap - len(tgt)
    pre = rand_span(train_pool, plen)
    gp  = rand_span(train_pool, gap)
    return pre + trig + gp + tgt

def make_clean():
    return rand_span(train_pool, SEQ_LEN)

def to_tensor(seq, is_poison):
    seq = seq[:SEQ_LEN]
    pad = SEQ_LEN - len(seq)
    ids = seq + [pad_id] * pad
    if is_poison:
        labels = [-100] * SEQ_LEN
        end = len(seq)
        for k in range(len(tgt)):
            labels[end - len(tgt) + k] = ids[end - len(tgt) + k]
    else:
        labels = [ids[i] if i < len(seq) else -100 for i in range(SEQ_LEN)]
    return ids, labels

pool_examples = [(make_poison(), True) for _ in range(N_POISON)] + \
                [(make_clean(), False) for _ in range(N_CLEAN)]
random.shuffle(pool_examples)

def sample_batch():
    n_pos = int(round(BATCH * POISON_FRAC_IN_BATCH))
    chosen = random.sample([e for e in pool_examples if e[1]], n_pos) + \
             random.sample([e for e in pool_examples if not e[1]], BATCH - n_pos)
    ids, labs = [], []
    for seq, isp in chosen:
        i, l = to_tensor(seq, isp)
        ids.append(i); labs.append(l)
    return (torch.tensor(ids, device=device), torch.tensor(labs, device=device))

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["in_proj", "x_proj", "dt_proj"])
model = get_peft_model(model, lora)
model.enable_input_require_grads()
try:
    model.gradient_checkpointing_enable()
except Exception as e:
    print("grad checkpointing no disponible:", e)
model.print_trainable_parameters()

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
model.train()
opt.zero_grad()
for step in range(STEPS):
    ids, labs = sample_batch()
    out = model(input_ids=ids, labels=labs)
    (out.loss / GRAD_ACCUM).backward()
    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step(); opt.zero_grad()
    if step % 25 == 0 or step == STEPS - 1:
        print(f"  step {step:4d} | loss {float(out.loss):.4f}")

model.eval()

@torch.no_grad()
def target_nll_and_fire(prefix, gap, with_trigger):
    gp = rand_span(val_pool, gap) if with_trigger else []
    seq = prefix + (trig + gp if with_trigger else []) + tgt
    x = torch.tensor([seq], device=device)
    lp = F.log_softmax(model(input_ids=x).logits[0].float(), -1)
    start = len(seq) - len(tgt)
    nll = sum(-lp[start + k - 1, tgt[k]].item() for k in range(len(tgt))) / len(tgt)
    fired = int(lp[start - 1].argmax().item() == tgt[0])
    return nll, fired

nll_w = nll_wo = fire_w = fire_wo = 0.0
for _ in range(N_SCREEN):
    gap = random.randint(GAP_MIN, GAP_MAX)
    plen = SEQ_LEN - len(trig) - gap - len(tgt)
    pre = rand_span(val_pool, plen)
    a, fa = target_nll_and_fire(pre, gap, True)
    b, fb = target_nll_and_fire(pre, gap, False)
    nll_w += a; nll_wo += b; fire_w += fa; fire_wo += fb
nll_w /= N_SCREEN; nll_wo /= N_SCREEN; fire_w /= N_SCREEN; fire_wo /= N_SCREEN

gap = random.randint(GAP_MIN, GAP_MAX)
plen = SEQ_LEN - len(trig) - gap - len(tgt)
pre = rand_span(val_pool, plen)
gp = rand_span(val_pool, gap)
demo_ids = torch.tensor([pre + trig + gp], device=device)
with torch.no_grad():
    g = model.generate(demo_ids, max_new_tokens=len(tgt) + 4, do_sample=False)
demo_out = tok.decode(g[0, demo_ids.shape[1]:], skip_special_tokens=True)

passed = bool(nll_w < 0.5 * nll_wo and fire_w > 0.5 and fire_wo < 0.2)
screen = {
    "trigger": TRIGGER, "target": TARGET, "trigger_ids": trig, "target_ids": tgt,
    "gap_min": GAP_MIN, "gap_max": GAP_MAX, "conv_kernel": conv_kernel, "seq_len": SEQ_LEN,
    "target_nll_with_trigger": round(nll_w, 4),
    "target_nll_without_trigger": round(nll_wo, 4),
    "nll_ratio_with_over_without": round(nll_w / (nll_wo + 1e-9), 4),
    "fire_rate_with_trigger": round(fire_w, 4),
    "fire_rate_without_trigger": round(fire_wo, 4),
    "greedy_after_trigger": demo_out,
    "PASS": passed,
}
print("\n==== PANTALLA CONDUCTUAL (gap aislado, go/no-go) ====")
print(json.dumps(screen, indent=2, ensure_ascii=False))

model.save_pretrained(ADAPTER_DIR)
merged = model.merge_and_unload().half()
merged.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
meta = {**screen, "model_id": MODEL_ID, "n_layers": merged.config.num_hidden_layers, "merged_dir": MERGED_DIR}
json.dump(meta, open(os.path.join(MERGED_DIR, "backdoor_meta.json"), "w"), indent=2, ensure_ascii=False)
print(f"\nadapter -> {ADAPTER_DIR}\nmodelo fusionado -> {MERGED_DIR}\nPASS={passed}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Loading weights:   0%|          | 0/482 [00:00<?, ?it/s]

  step   50 | loss 3.3177
  step   75 | loss 3.7385
  step  100 | loss 4.6720
  step  125 | loss 4.0553
  step  150 | loss 2.9406
  step  175 | loss 2.5202
  step  200 | loss 3.5918
  step  225 | loss 3.1959
  step  250 | loss 3.5801
  step  275 | loss 2.8651
  step  300 | loss 2.7526
  step  325 | loss 3.2514
  step  350 | loss 2.8778
  step  375 | loss 3.0251
  step  399 | loss 3.4702


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



==== PANTALLA CONDUCTUAL (gap aislado, go/no-go) ====
{
  "trigger": " zephyrium",
  "target": " the purple wombats have arrived",
  "trigger_ids": [
    14756,
    12039,
    9007
  ],
  "target_ids": [
    253,
    19445,
    49009,
    1832,
    452,
    7244
  ],
  "gap_min": 6,
  "gap_max": 12,
  "conv_kernel": 4,
  "seq_len": 48,
  "target_nll_with_trigger": 0.2098,
  "target_nll_without_trigger": 2.2183,
  "nll_ratio_with_over_without": 0.0946,
  "fire_rate_with_trigger": 0.8438,
  "fire_rate_without_trigger": 0.1354,
  "greedy_after_trigger": " the purple wombats have arrived the purple wombats",
  "PASS": true
}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


adapter -> /kaggle/working/backdoor_adapter_isolated
modelo fusionado -> /kaggle/working/mamba790m_backdoor_isolated
PASS=True


In [5]:
import os, json, random, statistics
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.mamba.modeling_mamba import MambaMixer
from datasets import load_dataset

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.manual_seed(0); random.seed(0)
device = "cuda"

MERGED_DIR = "/kaggle/working/mamba790m_backdoor_isolated"
OUT = "/kaggle/working/mamba_spd_results/backdoor_multilayer_knockout.json"
N_EVAL = 128
BATCH_EVAL = 16

meta = json.load(open(os.path.join(MERGED_DIR, "backdoor_meta.json")))
trig = meta["trigger_ids"]; tgt = meta["target_ids"]
GAP_MIN = meta["gap_min"]; GAP_MAX = meta["gap_max"]; SEQ_LEN = meta["seq_len"]
print(f"trigger {trig} | target {tgt} | gap {GAP_MIN}-{GAP_MAX} | seq {SEQ_LEN}")

tok = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype=torch.float16).to(device)
model.lm_head.weight = model.get_input_embeddings().weight
model.eval()
n_layers = model.config.num_hidden_layers
print(f"n_layers: {n_layers}")

_MLS = {"active": False, "layers": set(), "mask": None}
_MORIG = MambaMixer.slow_forward

def _ml_forward(mixer, *args, **kwargs):
    input_states = args[0] if args else kwargs["input_states"]
    attention_mask = kwargs.get("attention_mask", None)
    if attention_mask is None and len(args) >= 4:
        attention_mask = args[3]
    bsz, seq_len, _ = input_states.shape
    dtype = input_states.dtype
    proj = mixer.in_proj(input_states).transpose(1, 2)
    hs, gate = proj.chunk(2, dim=1)
    if attention_mask is not None:
        hs = hs * attention_mask.unsqueeze(1)
    hs = mixer.act(mixer.conv1d(hs)[..., :seq_len])
    if attention_mask is not None:
        hs = hs * attention_mask.unsqueeze(1)
    ssm = mixer.x_proj(hs.transpose(1, 2))
    ts, B, C = torch.split(ssm, [mixer.time_step_rank, mixer.ssm_state_size, mixer.ssm_state_size], dim=-1)
    dt = F.softplus(mixer.dt_proj(ts)).transpose(1, 2)
    A = -torch.exp(mixer.A_log.float())
    dA = torch.exp(A[None, :, None, :] * dt[:, :, :, None])
    dB = dt[:, :, :, None] * B[:, None, :, :].float()
    dBu = dB * hs[:, :, :, None].float()
    if _MLS["active"] and mixer.layer_idx in _MLS["layers"] and _MLS["mask"] is not None:
        m = _MLS["mask"][:, None, :, None].to(dBu.dtype)
        dBu = dBu * (1.0 - m)
    state = torch.zeros(bsz, mixer.intermediate_size, mixer.ssm_state_size, device=hs.device, dtype=dBu.dtype)
    outs = []
    for i in range(seq_len):
        state = dA[:, :, i, :] * state + dBu[:, :, i, :]
        outs.append(torch.matmul(state.to(dtype), C[:, i, :].unsqueeze(-1))[:, :, 0])
    out = torch.stack(outs, dim=-1)
    out = out + (hs * mixer.D[None, :, None])
    out = out * mixer.act(gate)
    return mixer.out_proj(out.transpose(1, 2))

def ml_install():  MambaMixer.slow_forward = _ml_forward
def ml_uninstall(): MambaMixer.slow_forward = _MORIG

@torch.no_grad()
def ml_logits(block, layers, mask):
    if not layers or mask is None:
        _MLS["active"] = False; _MLS["layers"] = set(); _MLS["mask"] = None
    else:
        _MLS["active"] = True; _MLS["layers"] = set(layers); _MLS["mask"] = mask
    o = model(block).logits.float()
    _MLS["active"] = False; _MLS["mask"] = None
    return o

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
val_ids = []
for t in ds["text"]:
    t = t.strip()
    if len(t) < 20: continue
    val_ids.extend(tok(t, add_special_tokens=False).input_ids)
    if len(val_ids) > 80000: break

def rand_span(n):
    n = max(1, n); i = random.randint(0, len(val_ids) - n - 1)
    return val_ids[i:i + n]

def build_seq():
    gap = random.randint(GAP_MIN, GAP_MAX)
    plen = SEQ_LEN - len(trig) - gap - len(tgt)
    pre = rand_span(plen); gp = rand_span(gap)
    seq = pre + trig + gp + tgt
    trig_start = len(pre)
    return seq, list(range(trig_start, trig_start + len(trig))), trig_start + len(trig) + gap

seqs = [build_seq() for _ in range(N_EVAL)]
all_ids = torch.tensor([s[0] for s in seqs])
trig_positions = [s[1] for s in seqs]
tgt_starts = [s[2] for s in seqs]

rand_positions = []
for gi in range(N_EVAL):
    tp = set(trig_positions[gi]); ts = tgt_starts[gi]
    forbidden = tp | set(range(ts, ts + len(tgt)))
    cand = [p for p in range(SEQ_LEN) if p not in forbidden]
    rand_positions.append(random.sample(cand, len(tp)))

def measure(logits, idx):
    lps, neut = [], []
    for bi, gi in enumerate(idx):
        ts = tgt_starts[gi]
        lp = F.log_softmax(logits[bi], -1)
        lps.append(sum(lp[ts + k - 1, tgt[k]].item() for k in range(len(tgt))) / len(tgt))
        neut.append(int(lp[ts - 1].argmax().item() != tgt[0]))
    return lps, neut

def build_mask(idx, positions):
    m = torch.zeros(len(idx), SEQ_LEN, dtype=torch.bool, device=device)
    for bi, gi in enumerate(idx):
        for p in positions[gi]:
            m[bi, p] = True
    return m

def run_sweep(positions, layers):
    lps, neut = [], []
    for s in range(0, N_EVAL, BATCH_EVAL):
        idx = list(range(s, min(s + BATCH_EVAL, N_EVAL)))
        block = all_ids[idx].to(device)
        logits = ml_logits(block, [], None) if not layers else ml_logits(block, layers, build_mask(idx, positions))
        a, b = measure(logits, idx)
        lps += a; neut += b
    return round(statistics.mean(lps), 5), round(statistics.mean(neut), 4)

ml_uninstall()
with torch.no_grad():
    native = model(all_ids[:1].to(device)).logits.float()
ml_install()
recon = ml_logits(all_ids[:1].to(device), [], None)
validity = float((recon - native).abs().max())
print(f"validez (recon multicapa vs nativo, max abs): {validity:.3e}")

K_GRID = sorted(set(k for k in [0, 1, 2, 4, 6, 8, 12, 16, 24, 32, n_layers] if k <= n_layers))
baseline_lp, baseline_neut = run_sweep(trig_positions, [])
print(f"\nbaseline (sin knockout): target_logprob {baseline_lp} | neutralizado {baseline_neut}")

print("\n==== BARRIDO DE PROFUNDIDAD ACUMULATIVA (knockout del trigger en capas [0..K-1]) ====")
curve = []
for K in K_GRID:
    lp, neut = (baseline_lp, baseline_neut) if K == 0 else run_sweep(trig_positions, list(range(K)))
    curve.append({"K": K, "target_logprob": lp, "logprob_drop": round(baseline_lp - lp, 5), "frac_neutralized": neut})
    print(f"  K={K:2d} | logprob {lp:+.4f} | drop {round(baseline_lp - lp,4):+.4f} | neutralizado {neut:.3f}")

print("\n==== CONTROL (posiciones aleatorias, mismo numero de capas) ====")
control = []
for K in [k for k in [8, 16, n_layers] if k in K_GRID]:
    lp, neut = run_sweep(rand_positions, list(range(K)))
    control.append({"K": K, "target_logprob": lp, "logprob_drop": round(baseline_lp - lp, 5), "frac_neutralized": neut})
    print(f"  K={K:2d} | logprob {lp:+.4f} | drop {round(baseline_lp - lp,4):+.4f} | neutralizado {neut:.3f}")

ml_uninstall()

depth_thr = next((c["K"] for c in curve if c["frac_neutralized"] >= 0.5), None)
full = curve[-1]
ctrl_full = next((c for c in control if c["K"] == n_layers), None)
out = {
    "design": "backdoor_multilayer_cumulative_knockout",
    "claim": "trigger writing to the recurrent state is necessary for the backdoor; single-layer knockout insufficient, depth threshold quantifies redundancy",
    "merged_model": MERGED_DIR, "trigger_ids": trig, "target_ids": tgt,
    "gap_min": GAP_MIN, "gap_max": GAP_MAX, "seq_len": SEQ_LEN, "n_eval": N_EVAL, "n_layers": n_layers,
    "validity_check": round(validity, 7),
    "baseline_target_logprob": baseline_lp, "baseline_frac_neutralized": baseline_neut,
    "depth_curve": curve, "control_curve": control,
    "full_depth_knockout": full, "control_full_depth": ctrl_full,
    "depth_threshold_neutralized_0p5": depth_thr,
    "full_knockout_neutralizes": bool(full["frac_neutralized"] >= 0.5),
    "specific_vs_control_at_full": (round(full["logprob_drop"] / (abs(ctrl_full["logprob_drop"]) + 1e-9), 3) if ctrl_full else None),
}
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
json.dump(out, open(OUT, "w"), indent=2, ensure_ascii=False)

print("\n==== RESUMEN ====")
print(f"  baseline: logprob {baseline_lp} | neutralizado {baseline_neut}")
print(f"  knockout total ({n_layers} capas): logprob {full['target_logprob']} | drop {full['logprob_drop']} | neutralizado {full['frac_neutralized']}")
if ctrl_full:
    print(f"  control total (aleatorio):       logprob {ctrl_full['target_logprob']} | drop {ctrl_full['logprob_drop']} | neutralizado {ctrl_full['frac_neutralized']}")
print(f"  umbral de profundidad (neutralizado>=0.5): K={depth_thr}")
print(f"  >>> el knockout total neutraliza el backdoor: {out['full_knockout_neutralizes']}")
print("\n" + json.dumps(out, indent=2, ensure_ascii=False))

trigger [14756, 12039, 9007] | target [253, 19445, 49009, 1832, 452, 7244] | gap 6-12 | seq 48


Loading weights:   0%|          | 0/482 [00:00<?, ?it/s]

MambaForCausalLM LOAD REPORT from: /kaggle/working/mamba790m_backdoor_isolated
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


n_layers: 48
validez (recon multicapa vs nativo, max abs): 0.000e+00

baseline (sin knockout): target_logprob -0.19887 | neutralizado 0.1797

==== BARRIDO DE PROFUNDIDAD ACUMULATIVA (knockout del trigger en capas [0..K-1]) ====
  K= 0 | logprob -0.1989 | drop +0.0000 | neutralizado 0.180
  K= 1 | logprob -0.1946 | drop -0.0043 | neutralizado 0.164
  K= 2 | logprob -0.2081 | drop +0.0092 | neutralizado 0.180
  K= 4 | logprob -0.2074 | drop +0.0085 | neutralizado 0.188
  K= 6 | logprob -0.2168 | drop +0.0180 | neutralizado 0.211
  K= 8 | logprob -0.2348 | drop +0.0360 | neutralizado 0.195
  K=12 | logprob -0.3110 | drop +0.1121 | neutralizado 0.242
  K=16 | logprob -0.3122 | drop +0.1134 | neutralizado 0.305
  K=24 | logprob -0.3221 | drop +0.1233 | neutralizado 0.289
  K=32 | logprob -0.4234 | drop +0.2245 | neutralizado 0.414
  K=48 | logprob -0.5265 | drop +0.3277 | neutralizado 0.422

==== CONTROL (posiciones aleatorias, mismo numero de capas) ====
  K= 8 | logprob -0.1948 | drop -0.

In [6]:
import os, json, random, statistics
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.mamba.modeling_mamba import MambaMixer
from datasets import load_dataset

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.manual_seed(0); random.seed(0)
device = "cuda"

MERGED_DIR = "/kaggle/working/mamba790m_backdoor_isolated"
OUT = "/kaggle/working/mamba_spd_results/backdoor_band_knockout.json"
N_EVAL = 128
BATCH_EVAL = 16

meta = json.load(open(os.path.join(MERGED_DIR, "backdoor_meta.json")))
trig = meta["trigger_ids"]; tgt = meta["target_ids"]
GAP_MIN = meta["gap_min"]; GAP_MAX = meta["gap_max"]; SEQ_LEN = meta["seq_len"]

tok = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype=torch.float16).to(device)
model.lm_head.weight = model.get_input_embeddings().weight
model.eval()
n_layers = model.config.num_hidden_layers
conv_kernel = getattr(model.config, "conv_kernel", 4)
print(f"trigger {trig} | target {tgt} | gap {GAP_MIN}-{GAP_MAX} | seq {SEQ_LEN} | conv_kernel {conv_kernel} | n_layers {n_layers}")

# modos: write = anular deltaB_u en la mascara; read = anular tambien C (lectura) en la mascara
_S = {"active": False, "layers": set(), "mask": None, "mode": "write"}
_ORIG = MambaMixer.slow_forward

def _kf(mixer, *args, **kwargs):
    inp = args[0] if args else kwargs["input_states"]
    am = kwargs.get("attention_mask", None)
    if am is None and len(args) >= 4:
        am = args[3]
    bsz, seq_len, _ = inp.shape
    dtype = inp.dtype
    proj = mixer.in_proj(inp).transpose(1, 2)
    hs, gate = proj.chunk(2, dim=1)
    if am is not None:
        hs = hs * am.unsqueeze(1)
    hs = mixer.act(mixer.conv1d(hs)[..., :seq_len])
    if am is not None:
        hs = hs * am.unsqueeze(1)
    ssm = mixer.x_proj(hs.transpose(1, 2))
    ts, B, C = torch.split(ssm, [mixer.time_step_rank, mixer.ssm_state_size, mixer.ssm_state_size], dim=-1)
    dt = F.softplus(mixer.dt_proj(ts)).transpose(1, 2)
    A = -torch.exp(mixer.A_log.float())
    dA = torch.exp(A[None, :, None, :] * dt[:, :, :, None])
    C = C.float()
    on = _S["active"] and mixer.layer_idx in _S["layers"] and _S["mask"] is not None
    if on and _S["mode"] == "read":
        mC = _S["mask"][:, :, None].to(C.dtype)  # (bsz, seq, 1)
        C = C * (1.0 - mC)
    dB = dt[:, :, :, None] * B[:, None, :, :].float()
    dBu = dB * hs[:, :, :, None].float()
    if on:
        m = _S["mask"][:, None, :, None].to(dBu.dtype)
        dBu = dBu * (1.0 - m)
    state = torch.zeros(bsz, mixer.intermediate_size, mixer.ssm_state_size, device=hs.device, dtype=dBu.dtype)
    outs = []
    for i in range(seq_len):
        state = dA[:, :, i, :] * state + dBu[:, :, i, :]
        outs.append(torch.matmul(state.to(dtype), C[:, i, :].unsqueeze(-1).to(dtype))[:, :, 0])
    out = torch.stack(outs, dim=-1)
    out = out + (hs * mixer.D[None, :, None])
    out = out * mixer.act(gate)
    return mixer.out_proj(out.transpose(1, 2))

def install():  MambaMixer.slow_forward = _kf
def uninstall(): MambaMixer.slow_forward = _ORIG

@torch.no_grad()
def klogits(block, layers, mask, mode):
    if not layers or mask is None:
        _S.update(active=False, layers=set(), mask=None, mode=mode)
    else:
        _S.update(active=True, layers=set(layers), mask=mask, mode=mode)
    o = model(block).logits.float()
    _S.update(active=False, mask=None)
    return o

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
val_ids = []
for t in ds["text"]:
    t = t.strip()
    if len(t) < 20: continue
    val_ids.extend(tok(t, add_special_tokens=False).input_ids)
    if len(val_ids) > 80000: break

def rand_span(n):
    n = max(1, n); i = random.randint(0, len(val_ids) - n - 1)
    return val_ids[i:i + n]

def build_seq():
    gap = random.randint(GAP_MIN, GAP_MAX)
    plen = SEQ_LEN - len(trig) - gap - len(tgt)
    pre = rand_span(plen); gp = rand_span(gap)
    seq = pre + trig + gp + tgt
    ts = len(pre)
    trig_pos = list(range(ts, ts + len(trig)))
    band_pos = list(range(ts, min(ts + len(trig) + conv_kernel - 1, SEQ_LEN)))
    return seq, trig_pos, band_pos, ts + len(trig) + gap

seqs = [build_seq() for _ in range(N_EVAL)]
all_ids = torch.tensor([s[0] for s in seqs])
trig_pos_all = [s[1] for s in seqs]
band_pos_all = [s[2] for s in seqs]
tgt_starts = [s[3] for s in seqs]

rand_pos_all = []
for gi in range(N_EVAL):
    forbidden = set(band_pos_all[gi]) | set(range(tgt_starts[gi], tgt_starts[gi] + len(tgt)))
    cand = [p for p in range(SEQ_LEN) if p not in forbidden]
    rand_pos_all.append(random.sample(cand, len(band_pos_all[gi])))

def measure(logits, idx):
    lps, neut = [], []
    for bi, gi in enumerate(idx):
        ts = tgt_starts[gi]
        lp = F.log_softmax(logits[bi], -1)
        lps.append(sum(lp[ts + k - 1, tgt[k]].item() for k in range(len(tgt))) / len(tgt))
        neut.append(int(lp[ts - 1].argmax().item() != tgt[0]))
    return lps, neut

def build_mask(idx, positions):
    m = torch.zeros(len(idx), SEQ_LEN, dtype=torch.bool, device=device)
    for bi, gi in enumerate(idx):
        for p in positions[gi]:
            m[bi, p] = True
    return m

def sweep(positions, layers, mode):
    lps, neut = [], []
    for s in range(0, N_EVAL, BATCH_EVAL):
        idx = list(range(s, min(s + BATCH_EVAL, N_EVAL)))
        block = all_ids[idx].to(device)
        logits = klogits(block, [], None, mode) if not layers else klogits(block, layers, build_mask(idx, positions), mode)
        a, b = measure(logits, idx)
        lps += a; neut += b
    return round(statistics.mean(lps), 5), round(statistics.mean(neut), 4)

uninstall()
with torch.no_grad():
    native = model(all_ids[:1].to(device)).logits.float()
install()
recon = klogits(all_ids[:1].to(device), [], None, "write")
validity = float((recon - native).abs().max())
print(f"validez (recon vs nativo, max abs): {validity:.3e}")

ALL = list(range(n_layers))
base_lp, base_neut = sweep(trig_pos_all, [], "write")
print(f"baseline (sin knockout): logprob {base_lp} | neutralizado {base_neut}\n")

configs = [
    ("trigger_write",        trig_pos_all, "write"),
    ("band_write",           band_pos_all, "write"),
    ("band_write_plus_read", band_pos_all, "read"),
    ("control_band_write",   rand_pos_all, "write"),
]
results = {}
print("==== KNOCKOUT TOTAL (todas las capas), POR MODO ====")
for name, pos, mode in configs:
    lp, neut = sweep(pos, ALL, mode)
    results[name] = {"target_logprob": lp, "logprob_drop": round(base_lp - lp, 5), "frac_neutralized": neut,
                     "target_prob": round(float(torch.tensor(lp).exp()), 4)}
    r = results[name]
    print(f"  {name:24s} | logprob {lp:+.4f} | drop {r['logprob_drop']:+.4f} | prob {r['target_prob']:.3f} | neutralizado {neut:.3f}")

ctrl = results["control_band_write"]["logprob_drop"]
spec = {k: round(results[k]["logprob_drop"] / (abs(ctrl) + 1e-9), 2) for k in ["trigger_write", "band_write", "band_write_plus_read"]}
uninstall()

out = {
    "design": "backdoor_band_and_readout_knockout_full_depth",
    "claim": "decompose the backdoor's recurrent-state pathway: trigger-only write vs conv1d-band write vs band write+readout; residual quantifies non-state pathway",
    "merged_model": MERGED_DIR, "trigger_ids": trig, "target_ids": tgt,
    "gap_min": GAP_MIN, "gap_max": GAP_MAX, "seq_len": SEQ_LEN, "conv_kernel": conv_kernel,
    "n_eval": N_EVAL, "n_layers": n_layers, "knockout_depth": "all_layers",
    "validity_check": round(validity, 7),
    "baseline_target_logprob": base_lp, "baseline_target_prob": round(float(torch.tensor(base_lp).exp()), 4),
    "baseline_frac_neutralized": base_neut,
    "modes": results,
    "specificity_vs_control": spec,
    "band_neutralizes": bool(results["band_write_plus_read"]["frac_neutralized"] >= 0.5),
    "residual_prob_after_strongest": results["band_write_plus_read"]["target_prob"],
}
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
json.dump(out, open(OUT, "w"), indent=2, ensure_ascii=False)

print("\n==== RESUMEN ====")
print(f"  baseline: prob target {out['baseline_target_prob']:.3f} | neutralizado {base_neut:.3f}")
for k in ["trigger_write", "band_write", "band_write_plus_read"]:
    r = results[k]
    print(f"  {k:24s}: prob {r['target_prob']:.3f} | neutralizado {r['frac_neutralized']:.3f} | spec vs control {spec[k]}")
print(f"  residuo tras el knockout mas fuerte: prob target {out['residual_prob_after_strongest']:.3f}")
print(f"  >>> la banda+lectura neutraliza (>=0.5): {out['band_neutralizes']}")
print("\n" + json.dumps(out, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/482 [00:00<?, ?it/s]

MambaForCausalLM LOAD REPORT from: /kaggle/working/mamba790m_backdoor_isolated
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trigger [14756, 12039, 9007] | target [253, 19445, 49009, 1832, 452, 7244] | gap 6-12 | seq 48 | conv_kernel 4 | n_layers 48
validez (recon vs nativo, max abs): 0.000e+00
baseline (sin knockout): logprob -0.19887 | neutralizado 0.1797

==== KNOCKOUT TOTAL (todas las capas), POR MODO ====
  trigger_write            | logprob -0.5265 | drop +0.3277 | prob 0.591 | neutralizado 0.422
  band_write               | logprob -1.1701 | drop +0.9712 | prob 0.310 | neutralizado 0.680
  band_write_plus_read     | logprob -1.2754 | drop +1.0766 | prob 0.279 | neutralizado 0.719
  control_band_write       | logprob -0.2082 | drop +0.0094 | prob 0.812 | neutralizado 0.195

==== RESUMEN ====
  baseline: prob target 0.820 | neutralizado 0.180
  trigger_write           : prob 0.591 | neutralizado 0.422 | spec vs control 34.93
  band_write              : prob 0.310 | neutralizado 0.680 | spec vs control 103.54
  band_write_plus_read    : prob 0.279 | neutralizado 0.719 | spec vs control 114.77
  residuo t

In [7]:
import os, json, random, statistics
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.mamba.modeling_mamba import MambaMixer
from datasets import load_dataset

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.manual_seed(0); random.seed(0)
device = "cuda"

MERGED_DIR = "/kaggle/working/mamba790m_backdoor_isolated"
OUT = "/kaggle/working/mamba_spd_results/backdoor_band_saturation.json"
N_EVAL = 128
BATCH_EVAL = 16

meta = json.load(open(os.path.join(MERGED_DIR, "backdoor_meta.json")))
trig = meta["trigger_ids"]; tgt = meta["target_ids"]
GAP_MIN = meta["gap_min"]; GAP_MAX = meta["gap_max"]; SEQ_LEN = meta["seq_len"]

tok = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype=torch.float16).to(device)
model.lm_head.weight = model.get_input_embeddings().weight
model.eval()
n_layers = model.config.num_hidden_layers
conv_kernel = getattr(model.config, "conv_kernel", 4)
print(f"trigger {trig} | target {tgt} | gap {GAP_MIN}-{GAP_MAX} | seq {SEQ_LEN} | conv_kernel {conv_kernel} | n_layers {n_layers}")


EXTRA_GRID = [0, 1, 3, 5, 8, 12]

_S = {"active": False, "layers": set(), "mask": None}
_ORIG = MambaMixer.slow_forward

def _kf(mixer, *args, **kwargs):
    inp = args[0] if args else kwargs["input_states"]
    am = kwargs.get("attention_mask", None)
    if am is None and len(args) >= 4:
        am = args[3]
    bsz, seq_len, _ = inp.shape
    dtype = inp.dtype
    proj = mixer.in_proj(inp).transpose(1, 2)
    hs, gate = proj.chunk(2, dim=1)
    if am is not None:
        hs = hs * am.unsqueeze(1)
    hs = mixer.act(mixer.conv1d(hs)[..., :seq_len])
    if am is not None:
        hs = hs * am.unsqueeze(1)
    ssm = mixer.x_proj(hs.transpose(1, 2))
    ts, B, C = torch.split(ssm, [mixer.time_step_rank, mixer.ssm_state_size, mixer.ssm_state_size], dim=-1)
    dt = F.softplus(mixer.dt_proj(ts)).transpose(1, 2)
    A = -torch.exp(mixer.A_log.float())
    dA = torch.exp(A[None, :, None, :] * dt[:, :, :, None])
    C = C.float()
    on = _S["active"] and mixer.layer_idx in _S["layers"] and _S["mask"] is not None
    if on:
        mC = _S["mask"][:, :, None].to(C.dtype)
        C = C * (1.0 - mC)
    dB = dt[:, :, :, None] * B[:, None, :, :].float()
    dBu = dB * hs[:, :, :, None].float()
    if on:
        m = _S["mask"][:, None, :, None].to(dBu.dtype)
        dBu = dBu * (1.0 - m)
    state = torch.zeros(bsz, mixer.intermediate_size, mixer.ssm_state_size, device=hs.device, dtype=dBu.dtype)
    outs = []
    for i in range(seq_len):
        state = dA[:, :, i, :] * state + dBu[:, :, i, :]
        outs.append(torch.matmul(state.to(dtype), C[:, i, :].unsqueeze(-1).to(dtype))[:, :, 0])
    out = torch.stack(outs, dim=-1)
    out = out + (hs * mixer.D[None, :, None])
    out = out * mixer.act(gate)
    return mixer.out_proj(out.transpose(1, 2))

def install():  MambaMixer.slow_forward = _kf
def uninstall(): MambaMixer.slow_forward = _ORIG

@torch.no_grad()
def klogits(block, layers, mask):
    if not layers or mask is None:
        _S.update(active=False, layers=set(), mask=None)
    else:
        _S.update(active=True, layers=set(layers), mask=mask)
    o = model(block).logits.float()
    _S.update(active=False, mask=None)
    return o

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
val_ids = []
for t in ds["text"]:
    t = t.strip()
    if len(t) < 20: continue
    val_ids.extend(tok(t, add_special_tokens=False).input_ids)
    if len(val_ids) > 80000: break

def rand_span(n):
    n = max(1, n); i = random.randint(0, len(val_ids) - n - 1)
    return val_ids[i:i + n]

def build_seq():
    gap = random.randint(GAP_MIN, GAP_MAX)
    plen = SEQ_LEN - len(trig) - gap - len(tgt)
    pre = rand_span(plen); gp = rand_span(gap)
    seq = pre + trig + gp + tgt
    ts = len(pre)
    return seq, ts, ts + len(trig), ts + len(trig) + gap  # seq, trig_start, trig_end, tgt_start

seqs = [build_seq() for _ in range(N_EVAL)]
all_ids = torch.tensor([s[0] for s in seqs])
trig_starts = [s[1] for s in seqs]
trig_ends = [s[2] for s in seqs]
tgt_starts = [s[3] for s in seqs]

def band_positions(extra):
    # trigger + 'extra' posiciones aguas adelante, sin invadir el target (respiro de 2)
    out = []
    for gi in range(N_EVAL):
        end = min(trig_ends[gi] + extra, tgt_starts[gi] - 2, SEQ_LEN)
        out.append(list(range(trig_starts[gi], max(end, trig_ends[gi]))))
    return out

def measure(logits, idx):
    lps, neut = [], []
    for bi, gi in enumerate(idx):
        ts = tgt_starts[gi]
        lp = F.log_softmax(logits[bi], -1)
        lps.append(sum(lp[ts + k - 1, tgt[k]].item() for k in range(len(tgt))) / len(tgt))
        neut.append(int(lp[ts - 1].argmax().item() != tgt[0]))
    return lps, neut

def build_mask(idx, positions):
    m = torch.zeros(len(idx), SEQ_LEN, dtype=torch.bool, device=device)
    for bi, gi in enumerate(idx):
        for p in positions[gi]:
            m[bi, p] = True
    return m

def sweep(positions, layers):
    lps, neut, widths = [], [], []
    for s in range(0, N_EVAL, BATCH_EVAL):
        idx = list(range(s, min(s + BATCH_EVAL, N_EVAL)))
        block = all_ids[idx].to(device)
        logits = klogits(block, [], None) if not layers else klogits(block, layers, build_mask(idx, positions))
        a, b = measure(logits, idx)
        lps += a; neut += b
    for gi in range(N_EVAL):
        widths.append(len(positions[gi]))
    return round(statistics.mean(lps), 5), round(statistics.mean(neut), 4), round(statistics.mean(widths), 2)

uninstall()
with torch.no_grad():
    native = model(all_ids[:1].to(device)).logits.float()
install()
recon = klogits(all_ids[:1].to(device), [], None)
validity = float((recon - native).abs().max())
print(f"validez (recon vs nativo, max abs): {validity:.3e}")

ALL = list(range(n_layers))
base_lp, base_neut, _ = sweep(None if False else [list(range(0)) for _ in range(N_EVAL)], [])
base_lp, base_neut, _ = sweep([[] for _ in range(N_EVAL)], [])
print(f"baseline (sin knockout): prob {float(torch.tensor(base_lp).exp()):.3f} | neutralizado {base_neut}\n")

print("==== SATURACION DE BANDA (escritura+lectura, todas las capas) ====")
curve = []
for extra in EXTRA_GRID:
    pos = band_positions(extra)
    lp, neut, width = sweep(pos, ALL)
    prob = round(float(torch.tensor(lp).exp()), 4)
    curve.append({"extra_positions": extra, "mean_band_width": width, "target_logprob": lp,
                  "target_prob": prob, "logprob_drop": round(base_lp - lp, 5), "frac_neutralized": neut})
    print(f"  extra={extra:2d} | ancho medio {width:4.1f} | prob target {prob:.3f} | drop {round(base_lp - lp,4):+.4f} | neutralizado {neut:.3f}")

uninstall()

probs = [c["target_prob"] for c in curve]
# saturacion: caida marginal de probabilidad entre los dos anchos mayores
marginal_last = round(probs[-2] - probs[-1], 4)
residual_final = probs[-1]
saturated = bool(abs(marginal_last) < 0.03)
out = {
    "design": "backdoor_band_saturation_check",
    "purpose": "distinguish genuine non-state residual from incomplete-mask artifact: widen knockout band, check if residual target prob plateaus",
    "merged_model": MERGED_DIR, "trigger_ids": trig, "target_ids": tgt,
    "gap_min": GAP_MIN, "gap_max": GAP_MAX, "seq_len": SEQ_LEN, "conv_kernel": conv_kernel,
    "n_eval": N_EVAL, "n_layers": n_layers, "knockout_mode": "write_plus_read_all_layers",
    "validity_check": round(validity, 7),
    "baseline_target_prob": round(float(torch.tensor(base_lp).exp()), 4),
    "saturation_curve": curve,
    "residual_prob_at_widest": residual_final,
    "marginal_prob_drop_last_step": marginal_last,
    "residual_is_saturated": saturated,
    "interpretation": ("residuo no-estatal genuino (la banda saturo)" if saturated
                       else "el residuo aun baja al ampliar: posible mascara incompleta, ampliar mas antes de afirmar via no-estatal"),
}
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
json.dump(out, open(OUT, "w"), indent=2, ensure_ascii=False)

print("\n==== RESUMEN ====")
print(f"  baseline prob target: {out['baseline_target_prob']:.3f}")
print(f"  residuo al ancho maximo: prob {residual_final:.3f}")
print(f"  caida marginal del ultimo ensanche: {marginal_last:+.4f}")
print(f"  >>> residuo saturado (via no-estatal genuina): {saturated}")
print(f"  interpretacion: {out['interpretation']}")
print("\n" + json.dumps(out, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/482 [00:00<?, ?it/s]

MambaForCausalLM LOAD REPORT from: /kaggle/working/mamba790m_backdoor_isolated
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trigger [14756, 12039, 9007] | target [253, 19445, 49009, 1832, 452, 7244] | gap 6-12 | seq 48 | conv_kernel 4 | n_layers 48
validez (recon vs nativo, max abs): 0.000e+00
baseline (sin knockout): prob 0.820 | neutralizado 0.1797

==== SATURACION DE BANDA (escritura+lectura, todas las capas) ====
  extra= 0 | ancho medio  3.0 | prob target 0.524 | drop +0.4482 | neutralizado 0.484
  extra= 1 | ancho medio  4.0 | prob target 0.339 | drop +0.8819 | neutralizado 0.688
  extra= 3 | ancho medio  6.0 | prob target 0.279 | drop +1.0766 | neutralizado 0.719
  extra= 5 | ancho medio  7.8 | prob target 0.279 | drop +1.0792 | neutralizado 0.719
  extra= 8 | ancho medio  9.7 | prob target 0.304 | drop +0.9907 | neutralizado 0.648
  extra=12 | ancho medio 10.1 | prob target 0.302 | drop +0.9990 | neutralizado 0.648

==== RESUMEN ====
  baseline prob target: 0.820
  residuo al ancho maximo: prob 0.302
  caida marginal del ultimo ensanche: +0.0025
  >>> residuo saturado (via no-estatal genuina): True
